In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from utils import preprocess_data, get_features_and_target
from classification_utils import (
    train_and_compare_classification,
    print_results_table,
    print_confusion_matrix
)

In [3]:
#подготовка данных
df = preprocess_data('data.xlsx')
X, y = get_features_and_target(df, 'SI',
                                task_type='classification',
                                threshold=None)

median_val = df['SI'].median()

print(f"\nЦелевая переменная: SI > медианы (медиана = {median_val:.3f})")
print(f"Размер X: {X.shape}, размер y: {y.shape}")
print(f"\nБаланс классов:")
print(f"  Класс 0 (SI <= медианы): {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
print(f"  Класс 1 (SI > медианы):  {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")

Загружено: 1001 объектов, 213 столбцов
Удалено константных признаков: 33
Пропуски заполнены медианой каждого столбца
Удалено признаков с корреляцией > 0.95: 33
Итого: 1001 объектов, 147 столбцов (144 признаков)

Целевая переменная: SI > медианы (медиана = 3.846)
Размер X: (1001, 144), размер y: (1001,)

Баланс классов:
  Класс 0 (SI <= медианы): 501 (50.0%)
  Класс 1 (SI > медианы):  500 (50.0%)


In [4]:
#Обученеие моделей
results_df, best_models, splits = train_and_compare_classification(
    X, y,
    target_name='SI > медианы',
    cv=5,
    test_size=0.3,
    random_state=42
)

X_train, X_test, y_train, y_test = splits

In [5]:
#Вывод таблицы
print_results_table(results_df, target_name='SI > медианы')

from pathlib import Path; Path('results').mkdir(parents=True, exist_ok=True)
results_df.to_csv('results/results_classification_SI_median.csv', index=False)


             Model    CV F1  Test Accuracy  Test Precision  Test Recall  Test F1  Test ROC_AUC
      RandomForest 0.650363       0.691030        0.705036     0.653333 0.678201      0.728455
               KNN 0.665219       0.674419        0.673333     0.673333 0.673333      0.737903
               SVC 0.672920       0.661130        0.664384     0.646667 0.655405      0.683775
  GradientBoosting 0.652606       0.674419        0.696970     0.613333 0.652482      0.709470
LogisticRegression 0.653120       0.651163        0.659574     0.620000 0.639175      0.680331
      DecisionTree 0.614586       0.611296        0.615385     0.586667 0.600683      0.644790

Лучшая модель: RandomForest (Test F1 = 0.6782)


In [6]:
#Визуализация и анализ лучшей модели
# Сравнение моделей по F1-score
plt.figure(figsize=(12, 6))
sorted_df = results_df.sort_values('Test F1', ascending=True)
plt.barh(sorted_df['Model'], sorted_df['Test F1'], color='steelblue', edgecolor='black')
plt.xlabel('F1-score на тестовой выборке')
plt.title('Сравнение моделей классификации по F1-score (SI > медианы)')
plt.axvline(x=0.5, color='red', linestyle='--', linewidth=1, label='Случайное угадывание')
plt.legend()
plt.tight_layout()
plt.savefig('results/comparison_classification_SI_median.png',
            dpi=100, bbox_inches='tight')
plt.close()

# Анализ лучшей модели
best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

y_pred_test = best_model.predict(X_test)

print_confusion_matrix(y_test, y_pred_test, model_name=best_model_name)

cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Класс 0', 'Класс 1'],
            yticklabels=['Класс 0', 'Класс 1'])
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title(f'Матрица ошибок: {best_model_name}\n(SI > медианы)')
plt.tight_layout()
plt.savefig('results/confusion_matrix_SI_median.png', dpi=100, bbox_inches='tight')
plt.close()


TN (правильно отрицательные): 110
FP (ложные срабатывания):     41
FN (пропущенные положительные): 52
TP (правильно положительные): 98
              precision    recall  f1-score   support

           0       0.68      0.73      0.70       151
           1       0.71      0.65      0.68       150

    accuracy                           0.69       301
   macro avg       0.69      0.69      0.69       301
weighted avg       0.69      0.69      0.69       301



In [7]:
#важность признаков
final_model = best_model.named_steps['model']
if hasattr(final_model, 'feature_importances_'):
    print(f"\nТоп-15 важных признаков для {best_model_name}")
    importances = pd.Series(final_model.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(15)
    for feat, imp in top_features.items():
        print(f"  {feat:30s}: {imp:.4f}")

    plt.figure(figsize=(10, 6))
    top_features.sort_values().plot(kind='barh', color='coral', edgecolor='black')
    plt.xlabel('Важность признака')
    plt.title(f'Топ-15 признаков для классификации SI > медианы\n({best_model_name})')
    plt.tight_layout()
    plt.savefig('results/feature_importance_SI_median.png',
                dpi=100, bbox_inches='tight')
    plt.close()


Топ-15 важных признаков для RandomForest
  MaxAbsEStateIndex             : 0.0249
  FractionCSP3                  : 0.0218
  MinEStateIndex                : 0.0201
  BCUT2D_MWLOW                  : 0.0201
  VSA_EState4                   : 0.0199
  BCUT2D_CHGLO                  : 0.0198
  qed                           : 0.0198
  BCUT2D_LOGPHI                 : 0.0187
  FpDensityMorgan3              : 0.0170
  SlogP_VSA5                    : 0.0164
  HallKierAlpha                 : 0.0157
  BCUT2D_MRLOW                  : 0.0154
  MolWt                         : 0.0153
  MaxAbsPartialCharge           : 0.0152
  VSA_EState3                   : 0.0146
